# Mecanismo de atención y autoatención con un ejemplo de juguete

## Sesión 6: Transformers / LLMs

En este notebook construiremos una intuición simple del mecanismo de atención y autoatención (*self-attention*).

La idea no es cubrir todos los detalles matemáticos del transformer, sino entender el flujo general:

1. partir desde representaciones de entrada,
2. construir **queries**, **keys** y **values**,
3. calcular puntajes de atención,
4. transformarlos en pesos con **softmax**,
5. combinar la información para obtener nuevas representaciones contextualizadas.

Trabajaremos con un ejemplo pequeño y totalmente numérico.

## Objetivo pedagógico

Al finalizar este notebook deberíamos entender, a nivel conceptual, que:

- cada token puede comparar su información con otros tokens,
- no todos los tokens reciben la misma importancia,
- la salida de self-attention es una nueva representación que incorpora contexto.

Este notebook es un puente entre las slides conceptuales y el uso de transformers preentrenados en tareas reales.

In [1]:
import numpy as np
import pandas as pd

np.set_printoptions(precision=3, suppress=True)

## Paso 1: definimos una secuencia pequeña

Usaremos una frase muy corta formada por 4 tokens.

A cada token le asignaremos un embedding pequeño de dimensión 5.
No importa que estos embeddings no provengan de un modelo real: aquí solo queremos entender el mecanismo.

In [ ]:
tokens = ["El", "gato", "come", "pescado"]

X = np.array([
    [1.0, 0.0, 1.0, 0.0, 1.0],   # El
    [0.0, 2.0, 0.0, 1.0, 0.0],   # gato
    [1.0, 1.0, 0.0, 1.0, 0.0],   # come
    [0.0, 1.0, 1.0, 0.0, 1.0]    # pescado
])

df_X = pd.DataFrame(X, index=tokens, columns=[f"d{i+1}" for i in range(X.shape[1])])
df_X

## Interpretación de la matriz de entrada

Cada fila corresponde a un token.
Cada columna corresponde a una dimensión del embedding.

En un transformer real, estos embeddings serían aprendidos o vendrían de un modelo preentrenado.
Aquí los estamos fijando manualmente para trabajar con un ejemplo simple.

## Paso 2: construimos matrices de proyección

En self-attention, cada embedding de entrada se transforma en tres nuevas representaciones:

- **Query (Q)**: qué está buscando este token,
- **Key (K)**: qué ofrece este token,
- **Value (V)**: qué información aporta este token.

Para eso usamos tres matrices de pesos:
- $W_Q$
- $W_K$
- $W_V$

En este ejemplo también las fijaremos manualmente.

In [ ]:
W_Q = np.array([
    [1.0, 0.0, 1.0],
    [0.0, 1.0, 0.0],
    [1.0, 0.0, 0.0],
    [0.0, 1.0, 1.0],
    [1.0, 0.0, 1.0]
])

W_K = np.array([
    [1.0, 1.0, 0.0],
    [0.0, 1.0, 1.0],
    [1.0, 0.0, 1.0],
    [0.0, 1.0, 0.0],
    [1.0, 0.0, 1.0]
])

W_V = np.array([
    [1.0, 0.0, 1.0],
    [0.0, 2.0, 0.0],
    [1.0, 0.0, 0.0],
    [0.0, 1.0, 1.0],
    [1.0, 1.0, 0.0]
])

print("W_Q shape:", W_Q.shape)
print("W_K shape:", W_K.shape)
print("W_V shape:", W_V.shape)

## ¿Por qué las matrices $W_Q$, $W_K$ y $W_V$ son de tamaño $5 \times 3$?

En este ejemplo, cada token parte con un embedding de dimensión 5.

Eso significa que cada fila de la matriz de entrada $X$ tiene 5 componentes.

Queremos proyectar cada embedding a un nuevo espacio de dimensión 3 para construir:

- queries,
- keys,
- values.

Por eso usamos matrices de pesos de tamaño $5 \times 3$:

$$
X \in \mathbb{R}^{4 \times 5}, \quad
W_Q, W_K, W_V \in \mathbb{R}^{5 \times 3}
$$

y así obtenemos:

$$
Q, K, V \in \mathbb{R}^{4 \times 3}
$$

La idea es simple:

- **5** = dimensión del embedding original,
- **3** = dimensión del nuevo espacio interno donde calculamos atención.

En este ejemplo se eligió 3 solo para mantener matrices pequeñas y fáciles de interpretar.

## Paso 3: calculamos Q, K y V

Multiplicamos la matriz de embeddings de entrada por cada matriz de proyección:

$$
Q = XW_Q,\quad K = XW_K,\quad V = XW_V
$$

Esto produce tres nuevas matrices, una para queries, otra para keys y otra para values.

In [ ]:
Q = X @ W_Q
K = X @ W_K
V = X @ W_V

print("Q shape:", Q.shape)
print("K shape:", K.shape)
print("V shape:", V.shape)

In [ ]:
df_Q = pd.DataFrame(Q, index=tokens, columns=[f"q{i+1}" for i in range(Q.shape[1])])
df_K = pd.DataFrame(K, index=tokens, columns=[f"k{i+1}" for i in range(K.shape[1])])
df_V = pd.DataFrame(V, index=tokens, columns=[f"v{i+1}" for i in range(V.shape[1])])

print("Queries (Q)")
display(df_Q)

print("Keys (K)")
display(df_K)

print("Values (V)")
display(df_V)

## Interpretación

Hasta aquí, cada token ya no está representado solo por su embedding original.

Ahora cada token tiene:

- una **query**,
- una **key**,
- un **value**.

La idea es que las queries se comparan con las keys para decidir a qué tokens conviene prestar atención.
Luego usamos los values para construir la salida.

## Paso 4: calculamos los puntajes de atención

Los puntajes de atención se calculan como:

$$
\text{scores} = \frac{QK^T}{\sqrt{d_k}}
$$

donde $d_k$ es la dimensión de las keys.

Esto nos da una matriz donde cada fila indica cuánto atiende un token a los demás.

In [ ]:
d_k = K.shape[1]
scores = (Q @ K.T) / np.sqrt(d_k)

df_scores = pd.DataFrame(scores, index=tokens, columns=tokens)
df_scores

## ¿Cómo leer esta matriz?

- Cada **fila** representa el token que está consultando.
- Cada **columna** representa el token al que se está mirando.
- Un valor más alto indica una mayor compatibilidad entre query y key.

Todavía no son probabilidades ni pesos finales.
Solo son puntajes.

## Paso 5: aplicamos softmax

Ahora transformamos los puntajes en pesos de atención.

Para eso aplicamos **softmax por fila**, de modo que cada fila sume 1.

Así obtenemos una distribución de atención para cada token.

In [ ]:
def softmax(x):
    exp_x = np.exp(x - np.max(x, axis=1, keepdims=True))
    return exp_x / np.sum(exp_x, axis=1, keepdims=True)

attention_weights = softmax(scores)

df_attention = pd.DataFrame(attention_weights, index=tokens, columns=tokens)
df_attention

In [ ]:
df_attention.sum(axis=1)

## Interpretación de los pesos de atención

Ahora sí tenemos una distribución interpretable:

- cada fila suma 1,
- cada número indica cuánta atención recibe cada token,
- un token puede atender más a sí mismo o a otros tokens.

Esto permite que cada palabra construya una representación usando contexto.

## Paso 6: construimos la salida de self-attention

La salida final se obtiene combinando los **values** con los pesos de atención:

$$
Z = \text{AttentionWeights} \cdot V
$$

Esto produce una nueva representación contextualizada para cada token.

In [ ]:
Z = attention_weights @ V

df_Z = pd.DataFrame(Z, index=tokens, columns=[f"z{i+1}" for i in range(Z.shape[1])])
df_Z

## ¿Qué significa esta salida?

La matriz $Z$ ya no corresponde a los embeddings originales.

Ahora cada token tiene una nueva representación que mezcla información de otros tokens,
ponderada según la atención aprendida.

Ese es el corazón de self-attention:
cada token se vuelve una representación **contextualizada**.

## Ejemplo: mirar una fila específica

Podemos observar, por ejemplo, cómo se construye la nueva representación del token **"come"**.

In [ ]:
token_idx = tokens.index("come")

print("Pesos de atención para 'come':")
print(df_attention.iloc[token_idx])

print("\nNueva representación de 'come':")
print(df_Z.iloc[token_idx])

## Interpretación del ejemplo

La nueva representación de **"come"** surge como una combinación ponderada de los values de todos los tokens.

En otras palabras:

- el token consulta el contexto,
- decide a qué tokens mirar más,
- y construye una nueva representación usando esa información.

## Paso 7: intuición de multi-head attention

En un transformer real no se usa solo una atención, sino varias en paralelo.

La idea es que distintas cabezas de atención pueden capturar distintos tipos de relaciones, por ejemplo:

- relaciones sintácticas,
- relaciones semánticas,
- dependencias cercanas,
- dependencias lejanas.

Luego esas salidas se combinan.

## Esquema conceptual de multi-head attention

1. Partimos desde la misma entrada.
2. Proyectamos varias veces a diferentes espacios Q, K y V.
3. Cada cabeza calcula su propia atención.
4. Las salidas se concatenan y se vuelven a proyectar.

Para esta sesión no necesitamos implementarlo completo.
Lo importante es quedarnos con la intuición.

## Conexión con transformers

Este mecanismo de self-attention es uno de los componentes centrales de los transformers.

Luego, un transformer completo agrega además:

- embeddings de posición,
- múltiples cabezas de atención,
- capas feed-forward,
- conexiones residuales,
- normalización.

Pero la intuición base empieza aquí:
**comparar, ponderar y combinar contexto**.

## Cierre

En este notebook vimos una versión simplificada de self-attention:

- partimos con embeddings de entrada,
- construimos Q, K y V,
- calculamos scores,
- aplicamos softmax,
- obtenemos nuevas representaciones contextualizadas.

Con esta idea ya estamos en condiciones de pasar a notebooks más aplicados con modelos transformer preentrenados.

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(6, 4))
plt.imshow(attention_weights, aspect="auto")
plt.xticks(range(len(tokens)), tokens, rotation=45)
plt.yticks(range(len(tokens)), tokens)
plt.title("Matriz de atención")
plt.colorbar()
plt.tight_layout()
plt.show()

## Pregunta de reflexión

¿Por qué podría ser útil que un token no dependa solo del token inmediatamente anterior, sino que pueda atender directamente a cualquier otra posición de la secuencia?